# Focused VSD CT-STL-target laterality audit

This notebook investigates the asymmetric X-only flags for `VSD_z023_Left` and `VSD_z036_Right` together with their contralateral knees. It is read-only with respect to raw CT, STL and legacy target data.

The decisive laterality test is four-bone local chirality, not absolute or relative LPS X alone. It uses femur, tibia, patella and fibula centroids and is therefore stable when a knee is rotated in the transverse plane. The result is checked independently in the source STL and the voxel target.

Success criteria: (1) all four STL sets align with their named CT crop, (2) STL and target chirality agree with the named side, (3) legacy target/predrr alignment remains above the existing recall and Dice floors, and (4) the cohort-wide multibone audit has 58/58 passing knees. Legacy data are evidence only; final certification still requires the versioned LPS outputs.

In [1]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import trimesh

ROOT = Path.cwd().resolve()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'data').exists(), 'Run from the project tree containing data/'

STL_ROOT = ROOT / 'data/external/ground_truth/VSD_ground_truth'
UPRIGHT_ROOT = ROOT / 'data/external/upright/VSD_cases'
LEGACY_GT_ROOT = ROOT / 'data/interim/gt_per_bone_256/healthy'
LEGACY_PREDRR_ROOT = ROOT / 'data/interim/predrr/healthy'
ALIGNMENT_CSV = LEGACY_GT_ROOT / 'gt_per_bone_alignment.csv'
REPORT_DIR = ROOT / 'reports/manifests'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BONES = ('femur', 'tibia', 'patella', 'fibula')
BONE_COLORS = {'femur': '#00bcd4', 'tibia': '#4caf50', 'patella': '#ff9800', 'fibula': '#f44336'}
FOCUS = ('VSD_z023_Left', 'VSD_z023_Right', 'VSD_z036_Left', 'VSD_z036_Right')
CT_SURFACE_HU = 150.0
MIN_INSIDE_FRACTION = 0.75
MIN_SURFACE_BONE_FRACTION = 0.40
MIN_RECALL = 0.80
MIN_DICE = 0.40
print('Project root:', ROOT)

Project root: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject


In [2]:
def split_sample(sample_id):
    return sample_id.rsplit('_', 1)


def stl_path(sample_id, bone):
    case, side = split_sample(sample_id)
    hits = [p for p in (STL_ROOT / case / side).glob('*.stl') if bone in p.name.lower()]
    assert len(hits) == 1, f'{sample_id} {bone}: expected one STL, found {hits}'
    return hits[0]


def source_ct_path(sample_id):
    case, _ = split_sample(sample_id)
    if case.startswith('VSD_z'):
        return UPRIGHT_ROOT / f'{sample_id}.nii.gz'
    return ROOT / 'data/raw/healthy' / case.replace('_', '.') / f'{sample_id}.nii.gz'


def load_mesh(sample_id, bone):
    return trimesh.load(stl_path(sample_id, bone), process=False)


def mesh_centroid(mesh):
    center = np.asarray(mesh.center_mass, dtype=float)
    if not np.all(np.isfinite(center)):
        center = np.asarray(mesh.vertices, dtype=float).mean(axis=0)
    return center


def target_centroid(sample_id, bone):
    path = LEGACY_GT_ROOT / sample_id / f'{sample_id}_{bone}.nii.gz'
    image = sitk.ReadImage(str(path))
    mask = sitk.GetArrayFromImage(image) > 0
    assert mask.any(), f'Empty target mask: {path}'
    z, y, x = np.argwhere(mask).mean(axis=0)
    return np.asarray(image.TransformContinuousIndexToPhysicalPoint((float(x), float(y), float(z))))


def local_chirality(centroids):
    superior = centroids['femur'] - centroids['tibia']
    anterior = centroids['patella'] - centroids['tibia']
    lateral = centroids['fibula'] - centroids['tibia']
    normal = np.cross(superior, anterior)
    raw = float(np.dot(lateral, normal) / (np.linalg.norm(lateral) * np.linalg.norm(normal)))
    # LPS +Z is superior. Normalize the known legacy VSD_001 S-I header reflection.
    si_reflected = bool(superior[2] < 0)
    score = -raw if si_reflected else raw
    inferred = 'Left' if score > 0 else 'Right'
    return raw, score, inferred, si_reflected


def surface_support(image, array_zyx, mesh, max_vertices=20000):
    vertices = np.asarray(mesh.vertices, dtype=float)
    vertices = vertices[::max(1, len(vertices) // max_vertices)]
    index = np.asarray([image.TransformPhysicalPointToContinuousIndex(tuple(map(float, point))) for point in vertices])
    size = np.asarray(image.GetSize(), dtype=float)
    inside = ((index >= 0) & (index < (size - 1))).all(axis=1)
    if not inside.any():
        return 0.0, 0.0
    ijk = np.rint(index[inside]).astype(int)
    hu = array_zyx[ijk[:, 2], ijk[:, 1], ijk[:, 0]]
    return float(inside.mean()), float((hu > CT_SURFACE_HU).mean())


def load_target_masks(sample_id):
    return {bone: sitk.GetArrayFromImage(sitk.ReadImage(str(LEGACY_GT_ROOT / sample_id / f'{sample_id}_{bone}.nii.gz'))) > 0 for bone in BONES}


def mesh_indices(image, mesh, max_vertices=8000):
    vertices = np.asarray(mesh.vertices, dtype=float)
    vertices = vertices[::max(1, len(vertices) // max_vertices)]
    return np.asarray([image.TransformPhysicalPointToContinuousIndex(tuple(map(float, point))) for point in vertices])

In [3]:
alignment = pd.read_csv(ALIGNMENT_CSV).set_index('key')
focus_rows = []
focus_cache = {}
for sample_id in FOCUS:
    case, named_side = split_sample(sample_id)
    meshes = {bone: load_mesh(sample_id, bone) for bone in BONES}
    stl_centroids = {bone: mesh_centroid(meshes[bone]) for bone in BONES}
    target_centroids = {bone: target_centroid(sample_id, bone) for bone in BONES}
    stl_raw, stl_score, stl_side, stl_si_reflected = local_chirality(stl_centroids)
    target_raw, target_score, target_side, target_si_reflected = local_chirality(target_centroids)

    ct_path = source_ct_path(sample_id)
    ct_image = sitk.ReadImage(str(ct_path))
    ct_array = sitk.GetArrayFromImage(ct_image).astype(np.float32)
    supports = {bone: surface_support(ct_image, ct_array, meshes[bone]) for bone in BONES}
    own_inside_min = min(value[0] for value in supports.values())
    own_bone_min = min(value[1] for value in supports.values())
    align = alignment.loc[sample_id]
    x_delta = float(stl_centroids['fibula'][0] - stl_centroids['tibia'][0])
    x_only_side = 'Left' if x_delta > 0 else 'Right'

    passed = (
        stl_side == named_side and target_side == named_side and
        own_inside_min >= MIN_INSIDE_FRACTION and own_bone_min >= MIN_SURFACE_BONE_FRACTION and
        float(align.recall) >= MIN_RECALL and float(align.dice_union_vs_predrr) >= MIN_DICE
    )
    focus_rows.append({
        'sample_id': sample_id, 'named_side': named_side,
        'x_only_inferred_side': x_only_side, 'fibula_minus_tibia_lps_x_mm': round(x_delta, 3),
        'stl_chirality_score': round(stl_score, 4), 'stl_inferred_side': stl_side,
        'target_chirality_score': round(target_score, 4), 'target_inferred_side': target_side,
        'stl_si_reflection_normalized': stl_si_reflected, 'target_si_reflection_normalized': target_si_reflected,
        'ct_stl_min_inside_fraction': round(own_inside_min, 4),
        'ct_stl_min_surface_bone_fraction': round(own_bone_min, 4),
        'target_predrr_recall': float(align.recall),
        'target_predrr_dice': float(align.dice_union_vs_predrr),
        'status': 'PASS' if passed else 'REVIEW_REQUIRED',
    })
    focus_cache[sample_id] = dict(image=ct_image, ct=ct_array, meshes=meshes, stl_centroids=stl_centroids)

focus_audit = pd.DataFrame(focus_rows)
focus_csv = REPORT_DIR / 'vsd_focused_ct_stl_target_overlay_v1.csv'
focus_audit.to_csv(focus_csv, index=False)
display(focus_audit)

,sample_id,named_side,x_only_inferred_side,fibula_minus_tibia_lps_x_mm,stl_chirality_score,stl_inferred_side,target_chirality_score,target_inferred_side,stl_si_reflection_normalized,target_si_reflection_normalized,ct_stl_min_inside_fraction,ct_stl_min_surface_bone_fraction,target_predrr_recall,target_predrr_dice,status
0,VSD_z023_Left,Left,Right,-6.997,0.7173,Left,0.7148,Left,False,False,0.8937,0.6163,0.9583,0.7522,PASS
1,VSD_z023_Right,Right,Right,-11.007,-0.6981,Right,-0.7004,Right,False,False,0.9394,0.5147,0.9209,0.7428,PASS
2,VSD_z036_Left,Left,Left,38.619,0.8338,Left,0.8345,Left,False,False,0.9331,0.6342,0.8452,0.6213,PASS
3,VSD_z036_Right,Right,Left,12.281,-0.7610,Right,-0.7605,Right,False,False,0.8001,0.5753,0.9081,0.6719,PASS


In [4]:
fig, axes = plt.subplots(len(FOCUS), 4, figsize=(18, 16), constrained_layout=True)
for row_index, sample_id in enumerate(FOCUS):
    cached = focus_cache[sample_id]
    image, ct, meshes = cached['image'], cached['ct'], cached['meshes']
    audit_row = focus_audit.set_index('sample_id').loc[sample_id]

    ax = axes[row_index, 0]
    ax.imshow(np.clip(ct, -450, 1050).max(axis=1), cmap='gray', origin='lower', aspect='auto')
    for bone in BONES:
        idx = mesh_indices(image, meshes[bone])
        ax.scatter(idx[:, 0], idx[:, 2], s=0.15, alpha=0.20, color=BONE_COLORS[bone])
    ax.set_title(f'{sample_id} native CT + STL (AP)')
    ax.axis('off')

    ax = axes[row_index, 1]
    ax.imshow(np.clip(ct, -450, 1050).max(axis=0), cmap='gray', origin='lower')
    for bone in BONES:
        idx = mesh_indices(image, meshes[bone])
        ax.scatter(idx[:, 0], idx[:, 1], s=0.15, alpha=0.20, color=BONE_COLORS[bone])
    tib = np.asarray(image.TransformPhysicalPointToContinuousIndex(tuple(map(float, cached['stl_centroids']['tibia']))))
    fib = np.asarray(image.TransformPhysicalPointToContinuousIndex(tuple(map(float, cached['stl_centroids']['fibula']))))
    ax.annotate('', xy=(fib[0], fib[1]), xytext=(tib[0], tib[1]), arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
    ax.set_title(f"X-only={audit_row.x_only_inferred_side}; chirality={audit_row.stl_inferred_side}")
    ax.axis('off')

    predrr = sitk.GetArrayFromImage(sitk.ReadImage(str(LEGACY_PREDRR_ROOT / f'{sample_id}.nii.gz'))).astype(np.float32)
    masks = load_target_masks(sample_id)
    union = np.logical_or.reduce([masks[bone] for bone in BONES])
    ax = axes[row_index, 2]
    ax.imshow(predrr.max(axis=1), cmap='gray', origin='lower')
    ax.contour(union.max(axis=1).astype(float), levels=[0.5], colors=['yellow'], linewidths=0.7, origin='lower')
    ax.set_title(f"Legacy target/predrr: recall={audit_row.target_predrr_recall:.3f}, Dice={audit_row.target_predrr_dice:.3f}")
    ax.axis('off')

    rgb = np.zeros((*union.max(axis=1).shape, 3), dtype=np.float32)
    color_rgb = {'femur': (0.0, 0.75, 0.85), 'tibia': (0.25, 0.8, 0.3), 'patella': (1.0, 0.55, 0.0), 'fibula': (0.95, 0.15, 0.15)}
    for bone in BONES:
        projection = masks[bone].max(axis=1)
        for channel, value in enumerate(color_rgb[bone]):
            rgb[..., channel] = np.maximum(rgb[..., channel], projection * value)
    ax = axes[row_index, 3]
    ax.imshow(rgb, origin='lower')
    ax.set_title(f"Four-bone target: {audit_row.target_inferred_side} ({audit_row.status})")
    ax.axis('off')

fig.suptitle('Focused VSD CT-STL-target overlay audit', fontsize=16)
figure_path = REPORT_DIR / 'vsd_focused_ct_stl_target_overlay_v1.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.close(fig)
print('Saved:', figure_path)

Saved: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\reports\manifests\vsd_focused_ct_stl_target_overlay_v1.png


In [5]:
cohort_rows = []
for case_dir in sorted(p for p in STL_ROOT.iterdir() if p.is_dir() and not p.name.startswith('_')):
    for side in ('Left', 'Right'):
        side_dir = case_dir / side
        if not side_dir.exists():
            continue
        sample_id = f'{case_dir.name}_{side}'
        centers = {bone: mesh_centroid(load_mesh(sample_id, bone)) for bone in BONES}
        raw, score, inferred, si_reflected = local_chirality(centers)
        cohort_rows.append({
            'sample_id': sample_id, 'named_side': side,
            'chirality_score_raw': round(raw, 4), 'chirality_score_si_normalized': round(score, 4),
            'inferred_side': inferred, 'si_reflection_normalized': si_reflected,
            'status': 'PASS' if inferred == side else 'REVIEW_REQUIRED',
            'method': 'four_bone_local_chirality_stl',
        })
cohort_audit = pd.DataFrame(cohort_rows).sort_values('sample_id').reset_index(drop=True)
cohort_csv = REPORT_DIR / 'vsd_cohort_laterality_multibone_v2.csv'
cohort_audit.to_csv(cohort_csv, index=False)
cohort_summary = {
    'audit_version': 'vsd_cohort_laterality_multibone_v2',
    'method': 'four_bone_local_chirality_stl_with_si_reflection_normalization',
    'supersedes': 'vsd_cohort_laterality_intrinsic_v1 (fibula-minus-tibia LPS X only)',
    'total': int(len(cohort_audit)),
    'pass': int((cohort_audit.status == 'PASS').sum()),
    'review_required': int((cohort_audit.status != 'PASS').sum()),
    'focus_samples': focus_audit[['sample_id', 'status']].to_dict('records'),
    'limitation': 'Computed from current STL and legacy target evidence; final target certification waits for versioned LPS replay.',
}
cohort_json = REPORT_DIR / 'vsd_cohort_laterality_multibone_v2.json'
cohort_json.write_text(json.dumps(cohort_summary, indent=2), encoding='utf-8')
display(cohort_audit[cohort_audit.sample_id.str.contains('z023|z036', regex=True)])
print(json.dumps(cohort_summary, indent=2))

,sample_id,named_side,chirality_score_raw,chirality_score_si_normalized,inferred_side,si_reflection_normalized,status,method
30,VSD_z023_Left,Left,0.7173,0.7173,Left,False,PASS,four_bone_local_chirality_stl
31,VSD_z023_Right,Right,-0.6981,-0.6981,Right,False,PASS,four_bone_local_chirality_stl
36,VSD_z036_Left,Left,0.8338,0.8338,Left,False,PASS,four_bone_local_chirality_stl
37,VSD_z036_Right,Right,-0.7610,-0.7610,Right,False,PASS,four_bone_local_chirality_stl


{
  "audit_version": "vsd_cohort_laterality_multibone_v2",
  "method": "four_bone_local_chirality_stl_with_si_reflection_normalization",
  "supersedes": "vsd_cohort_laterality_intrinsic_v1 (fibula-minus-tibia LPS X only)",
  "total": 58,
  "pass": 58,
  "review_required": 0,
  "focus_samples": [
    {
      "sample_id": "VSD_z023_Left",
      "status": "PASS"
    },
    {
      "sample_id": "VSD_z023_Right",
      "status": "PASS"
    },
    {
      "sample_id": "VSD_z036_Left",
      "status": "PASS"
    },
    {
      "sample_id": "VSD_z036_Right",
      "status": "PASS"
    }
  ],
  "limitation": "Computed from current STL and legacy target evidence; final target certification waits for versioned LPS replay."
}


In [6]:
assert len(focus_audit) == 4
assert (focus_audit.status == 'PASS').all(), focus_audit
assert len(cohort_audit) == 58, f'Expected 58 retained VSD knees, found {len(cohort_audit)}'
assert (cohort_audit.status == 'PASS').all(), cohort_audit[cohort_audit.status != 'PASS']
evidence = {
    'audit_version': 'vsd_focused_ct_stl_target_overlay_v1',
    'verdict': 'PASS',
    'conclusion': 'The z023 Left and z036 Right X-only flags were false positives. Both sides of both subjects pass CT-STL support, four-bone STL/target chirality and target-predrr alignment. No mapping override is justified.',
    'focus_csv': focus_csv.name,
    'focus_csv_sha256': hashlib.sha256(focus_csv.read_bytes()).hexdigest(),
    'figure': figure_path.name,
    'figure_sha256': hashlib.sha256(figure_path.read_bytes()).hexdigest(),
    'cohort_csv': cohort_csv.name,
    'cohort_csv_sha256': hashlib.sha256(cohort_csv.read_bytes()).hexdigest(),
}
evidence_path = REPORT_DIR / 'vsd_focused_ct_stl_target_overlay_v1.json'
evidence_path.write_text(json.dumps(evidence, indent=2), encoding='utf-8')
print(json.dumps(evidence, indent=2))

{
  "audit_version": "vsd_focused_ct_stl_target_overlay_v1",
  "verdict": "PASS",
  "conclusion": "The z023 Left and z036 Right X-only flags were false positives. Both sides of both subjects pass CT-STL support, four-bone STL/target chirality and target-predrr alignment. No mapping override is justified.",
  "focus_csv": "vsd_focused_ct_stl_target_overlay_v1.csv",
  "focus_csv_sha256": "80f5f3c9031d84b25d0146c484053be9fe62ebea1e8d20e1e673e5f9d8a4092e",
  "figure": "vsd_focused_ct_stl_target_overlay_v1.png",
  "figure_sha256": "9da92942e83fc75cc86e140ef6da3c352a5509199c3f4a74038e9439e907d182",
  "cohort_csv": "vsd_cohort_laterality_multibone_v2.csv",
  "cohort_csv_sha256": "cc6fbe5bc344a3a35475f118d68cb55265ec55fcf869adc040c7b8e33c516718"
}
